In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp04/ex_7.py ──
"""
Shared infrastructure for MLFP04 Exercise 7 — Recommender Systems.

Scenario: a fictional Singapore electronics marketplace.
  - 300 users x 120 SKUs, explicit 1-5 star ratings
  - Ratings follow a known generative model so every technique can be
    checked against the truth:
        rating = global mean + user bias + item bias
                 + (user taste vector . item trait vector) + noise
    with N_LATENT_TRUE = 5 hidden taste dimensions.
  - 30% of (user, item) pairs are observed
  - 30% of observed ratings are held out for evaluation (~10 per user, so
    precision@5 and MAP actually have to RANK items)
  - N_COLD_ITEMS brand-new SKUs have ALL their ratings in the holdout set:
    they are cold-start items that no collaborative method has seen
  - A validation split (VAL_FRAC of training ratings) is carved out for
    tuning hybrid blend weights — the holdout set is never used for tuning
  - Item content features = a noisy linear view of each SKU's hidden traits
    and quality (think price tier, category, spec sheet)

Contains: synthetic rating matrix generation, baseline predictors, evaluation
metrics (RMSE, precision@k, MAP), shared constants, and output-dir setup.

Technique-specific code (content-based profile, user/item CF, ALS, hybrid
blending) lives in the per-technique files.
"""

from pathlib import Path

import numpy as np
import polars as pl


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex7_recommenders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS — synthetic SG e-commerce dataset
# ════════════════════════════════════════════════════════════════════════

N_USERS = 300
N_ITEMS = 120
N_LATENT_TRUE = 5
N_ITEM_FEATURES = 8
SPARSITY = 0.30  # fraction of (user, item) pairs that carry a rating
HOLDOUT_FRAC = 0.30  # fraction of observed ratings held out for evaluation
VAL_FRAC = 0.20  # fraction of TRAINING ratings reserved for blend tuning
N_COLD_ITEMS = 10  # brand-new SKUs whose every rating is in the holdout
RNG_SEED = 42

RATING_MIN = 1.0
RATING_MAX = 5.0
RELEVANCE_THRESHOLD = 3.5  # ratings >= 3.5 are "relevant" for ranking metrics

# Generative-model scales (documented so students can reason about them)
_GLOBAL_MEAN = 3.2
_USER_BIAS_SD = 0.35
_ITEM_BIAS_SD = 0.45
_TASTE_SCALE = 0.9
_NOISE_SD = 0.35
_FEATURE_NOISE_SD = 0.5


# ════════════════════════════════════════════════════════════════════════
# SYNTHETIC RATING MATRIX — shared across every technique
# ════════════════════════════════════════════════════════════════════════


def build_rating_dataset(
    seed: int = RNG_SEED,
) -> dict:
    """Generate the shared synthetic e-commerce rating matrix.

    Returns a dict with:
      R_observed       — (N_USERS, N_ITEMS) with NaN where not rated
      R_train          — R_observed with holdout entries set to NaN
      mask             — boolean observed mask
      train_mask       — boolean training-only mask (holdout removed)
      holdout_mask     — boolean holdout (test) mask
      fit_mask         — train_mask minus the validation ratings
      val_mask         — validation ratings (subset of train_mask), used ONLY
                         for tuning blend weights in 05_hybrid_evaluation.py
      cold_items       — boolean (N_ITEMS,) — SKUs with zero training ratings
      U_true, V_true   — ground-truth latent taste / trait factors
      item_features    — (N_ITEMS, N_ITEM_FEATURES) content-feature matrix
      user_ids, item_ids — stable string IDs
      ratings_df       — long-format polars DataFrame of observed ratings
      rng              — the generator, for downstream reproducible draws
    """
    rng = np.random.default_rng(seed=seed)

    U_true = rng.normal(0, 1, size=(N_USERS, N_LATENT_TRUE))
    V_true = rng.normal(0, 1, size=(N_ITEMS, N_LATENT_TRUE))
    user_bias = rng.normal(0, _USER_BIAS_SD, size=N_USERS)
    item_bias = rng.normal(0, _ITEM_BIAS_SD, size=N_ITEMS)

    taste = U_true @ V_true.T / np.sqrt(N_LATENT_TRUE)  # ~N(0, 1) per pair
    R_full = (
        _GLOBAL_MEAN
        + user_bias[:, None]
        + item_bias[None, :]
        + _TASTE_SCALE * taste
        + rng.normal(0, _NOISE_SD, size=(N_USERS, N_ITEMS))
    )
    R_full = np.clip(R_full, RATING_MIN, RATING_MAX)

    mask = rng.random(size=(N_USERS, N_ITEMS)) < SPARSITY
    R_observed = np.where(mask, R_full, np.nan)

    cold_items = np.zeros(N_ITEMS, dtype=bool)
    cold_items[rng.choice(N_ITEMS, size=N_COLD_ITEMS, replace=False)] = True

    holdout_mask = mask & (rng.random(size=(N_USERS, N_ITEMS)) < HOLDOUT_FRAC)
    holdout_mask |= mask & cold_items[None, :]  # cold SKUs: every rating is test
    train_mask = mask & ~holdout_mask
    R_train = np.where(train_mask, R_observed, np.nan)

    val_mask = train_mask & (rng.random(size=(N_USERS, N_ITEMS)) < VAL_FRAC)
    fit_mask = train_mask & ~val_mask

    # Content features: a noisy linear view of each item's hidden traits
    # AND its quality (item bias). Real catalogues behave the same way —
    # spec sheets and price tiers correlate with, but do not equal, what
    # drives ratings.
    hidden = np.hstack(
        [V_true / np.sqrt(N_LATENT_TRUE), (item_bias / _ITEM_BIAS_SD)[:, None]]
    )
    W_feat = rng.normal(0, 1, size=(N_LATENT_TRUE + 1, N_ITEM_FEATURES))
    item_features = hidden @ W_feat / np.sqrt(N_LATENT_TRUE + 1) + rng.normal(
        0, _FEATURE_NOISE_SD, size=(N_ITEMS, N_ITEM_FEATURES)
    )

    user_ids = [f"sg_user_{i:03d}" for i in range(N_USERS)]
    item_ids = [f"sku_{j:03d}" for j in range(N_ITEMS)]

    rows = []
    for i in range(N_USERS):
        for j in range(N_ITEMS):
            if mask[i, j]:
                rows.append(
                    {
                        "user_id": user_ids[i],
                        "item_id": item_ids[j],
                        "rating": round(float(R_observed[i, j]), 1),
                        "in_holdout": bool(holdout_mask[i, j]),
                    }
                )
    ratings_df = pl.DataFrame(rows)

    return {
        "R_observed": R_observed,
        "R_train": R_train,
        "mask": mask,
        "train_mask": train_mask,
        "holdout_mask": holdout_mask,
        "fit_mask": fit_mask,
        "val_mask": val_mask,
        "cold_items": cold_items,
        "U_true": U_true,
        "V_true": V_true,
        "item_features": item_features,
        "user_ids": user_ids,
        "item_ids": item_ids,
        "ratings_df": ratings_df,
        "rng": rng,
    }


# ════════════════════════════════════════════════════════════════════════
# BASELINES — every recommender must beat these to be worth deploying
# ════════════════════════════════════════════════════════════════════════


def baseline_predictions(
    R: np.ndarray, obs_mask: np.ndarray, seed: int = RNG_SEED
) -> dict[str, np.ndarray]:
    """Return three no-skill reference predictors fitted on ``R[obs_mask]``.

    - "Global mean":  every pair gets the mean training rating (RMSE floor)
    - "Item mean":    each SKU's mean training rating (popularity), falling
                      back to the global mean for SKUs with no training data
    - "Random":       uniform scores in [1, 5] — the chance line for P@k/MAP
    """
    global_mean = float(R[obs_mask].mean())
    n_users, n_items = R.shape
    counts = obs_mask.sum(axis=0)
    sums = np.where(obs_mask, R, 0.0).sum(axis=0)
    item_mean = np.where(counts > 0, sums / np.maximum(counts, 1), global_mean)
    rng = np.random.default_rng(seed)
    return {
        "Global mean": np.full((n_users, n_items), global_mean),
        "Item mean": np.tile(item_mean, (n_users, 1)),
        "Random": rng.uniform(RATING_MIN, RATING_MAX, size=(n_users, n_items)),
    }


# ════════════════════════════════════════════════════════════════════════
# EVALUATION METRICS — shared across every technique
# ════════════════════════════════════════════════════════════════════════


def holdout_rmse(
    predictions: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
) -> tuple[float, float]:
    """Return (RMSE, coverage) on the holdout mask.

    Coverage = fraction of holdout pairs where the method produced a
    non-NaN prediction. Methods that can't predict cold pairs have
    coverage < 1.0; their RMSE is computed only on the pairs they cover.
    """
    n_holdout = int(holdout_mask.sum())
    covered = holdout_mask & ~np.isnan(predictions)
    if not covered.any():
        return float("inf"), 0.0
    errors = (R_true[covered] - predictions[covered]) ** 2
    rmse = float(np.sqrt(np.mean(errors)))
    coverage = float(covered.sum()) / max(n_holdout, 1)
    return rmse, coverage


def precision_at_k(
    predictions: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
    k: int = 5,
    threshold: float = RELEVANCE_THRESHOLD,
) -> float:
    """Precision@k averaged across users.

    For each user, rank holdout items by predicted score, take the top-k,
    and compute what fraction of those have true rating >= threshold.
    Items a method cannot score (NaN) are never recommended.
    """
    precisions = []
    for u in range(predictions.shape[0]):
        holdout_items = np.where(holdout_mask[u])[0]
        if len(holdout_items) == 0:
            continue
        relevant = {j for j in holdout_items if R_true[u, j] >= threshold}
        if not relevant:
            continue
        scored = [
            (j, predictions[u, j])
            for j in holdout_items
            if not np.isnan(predictions[u, j])
        ]
        scored.sort(key=lambda x: -x[1])
        top = [j for j, _ in scored[:k]]
        if not top:
            precisions.append(0.0)
            continue
        precisions.append(len(set(top) & relevant) / len(top))
    return float(np.mean(precisions)) if precisions else 0.0


def mean_average_precision(
    predictions: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
    threshold: float = RELEVANCE_THRESHOLD,
) -> float:
    """Mean Average Precision across users on the holdout set.

    A relevant item the method cannot score counts as never retrieved, so
    low coverage lowers MAP.
    """
    aps = []
    for u in range(predictions.shape[0]):
        holdout_items = np.where(holdout_mask[u])[0]
        if len(holdout_items) == 0:
            continue
        relevant = {j for j in holdout_items if R_true[u, j] >= threshold}
        if not relevant:
            continue
        scored = [
            (j, predictions[u, j])
            for j in holdout_items
            if not np.isnan(predictions[u, j])
        ]
        scored.sort(key=lambda x: -x[1])

        hits = 0
        sum_precision = 0.0
        for rank, (j, _) in enumerate(scored, 1):
            if j in relevant:
                hits += 1
                sum_precision += hits / rank
        aps.append(sum_precision / len(relevant))
    return float(np.mean(aps)) if aps else 0.0


def evaluate_method(
    preds: np.ndarray, R_true: np.ndarray, holdout_mask: np.ndarray
) -> dict:
    """Return {"RMSE", "Coverage", "P@5", "MAP"} for one prediction matrix."""
    rmse, cov = holdout_rmse(preds, R_true, holdout_mask)
    return {
        "RMSE": rmse,
        "Coverage": cov,
        "P@5": precision_at_k(preds, R_true, holdout_mask, k=5),
        "MAP": mean_average_precision(preds, R_true, holdout_mask),
    }


def print_method_scores(
    name: str, preds: np.ndarray, R_true: np.ndarray, holdout_mask: np.ndarray
) -> dict:
    """Print a single-line scorecard and return the metrics dict."""
    m = evaluate_method(preds, R_true, holdout_mask)
    print(
        f"  {name:<22} RMSE={m['RMSE']:6.4f}  coverage={m['Coverage']:6.1%}  "
        f"P@5={m['P@5']:6.4f}  MAP={m['MAP']:6.4f}"
    )
    return m


def print_baselines(
    R_train: np.ndarray,
    train_mask: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
) -> dict[str, dict]:
    """Score the three no-skill baselines on the holdout and print them."""
    print("  Baselines (a recommender must beat these):")
    return {
        name: print_method_scores(name, preds, R_true, holdout_mask)
        for name, preds in baseline_predictions(R_train, train_mask).items()
    }


def print_warm_comparison(
    name: str,
    preds: np.ndarray,
    R_train: np.ndarray,
    train_mask: np.ndarray,
    R_true: np.ndarray,
    holdout_mask: np.ndarray,
    cold_items: np.ndarray,
) -> tuple[float, float]:
    """Compare a CF-style method with the item-mean baseline on WARM SKUs.

    Collaborative methods cannot score cold-start SKUs, and MAP counts an
    unscored relevant item as a miss — so on the full holdout they can lose
    to a baseline that covers everything. Restricting to warm SKUs isolates
    ranking skill from coverage. Returns (method MAP, item-mean MAP).
    """
    warm = holdout_mask & ~cold_items[None, :]
    method_map = mean_average_precision(preds, R_true, warm)
    item_mean = baseline_predictions(R_train, train_mask)["Item mean"]
    base_map = mean_average_precision(item_mean, R_true, warm)
    verdict = "beats" if method_map > base_map else "does NOT beat"
    print(
        f"  Warm SKUs only: {name} MAP={method_map:.4f} vs item-mean "
        f"MAP={base_map:.4f} -> {name} {verdict} the popularity baseline "
        "on items it can see."
    )
    return method_map, base_map


# ════════════════════════════════════════════════════════════════════════
# VISUAL HELPERS
# ════════════════════════════════════════════════════════════════════════


def save_html(fig, filename: str) -> Path:
    """Write a plotly fig into OUTPUT_DIR and return the path."""
    path = OUTPUT_DIR / filename
    fig.write_html(str(path))
    print(f"  saved: {path}")
    return path




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 7.4: Matrix Factorisation with ALS (from scratch)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Implement biased Alternating Least Squares (ALS) matrix factorisation
#   - Recover true latent structure from a sparse rating matrix
#   - Track convergence — verify the regularised loss is non-increasing
#   - Visualise learned user + item embeddings in 2D
#   - Understand the SVD++ extension for implicit feedback
#   - Articulate THE PIVOT: optimisation drives feature discovery
#
# PREREQUISITES: Exercises 7.1-7.3; MLFP04 Ex 3 (PCA/SVD)
#
# ESTIMATED TIME: ~45 min
#
# TASKS:
#   1. Theory — why R ≈ mu + b_u + b_j + U V^T and why ALS converges
#   2. Build — ALS update equations from scratch
#   3. Train — run the alternating least squares loop
#   4. Visualise — user + item embeddings (PCA projection) + convergence
#   5. Apply — music-streaming recommendation at scale
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import polars as pl
from kailash_ml import ModelVisualizer
from sklearn.decomposition import PCA


K_LATENT = N_LATENT_TRUE  # we search for as many factors as the data has
LAMBDA_REG = 5.0
N_ITERATIONS = 30



## THEORY — Why matrix factorisation works


In [ ]:
# The central assumption: every user and every item can be represented by
# a short vector of latent factors (k = 5 here), plus a per-user and a
# per-item bias. A rating is the global mean plus both biases plus the
# inner product of the user's factors and the item's factors:
#
#   R[u, j] ≈ mu + b_u + b_j + U[u] · V[j]
#
# The biases soak up "this user is generous" and "this SKU is simply good";
# the factors capture taste — WHICH kinds of items this user likes.
#
# If we could see the full R, SVD would give us U and V in closed form.
# Since R is sparse (only ~30% observed), we minimise a regularised loss:
#
#   L = sum_{(u,j) observed} (R[u,j] - mu - b_u - b_j - U[u] · V[j])^2
#     + lambda * (||U||^2 + ||V||^2 + ||b_user||^2 + ||b_item||^2)
#
# ALS: fix the item side, solve for each user's [U[u], b_u] (a ridge
# regression). Then fix the user side, solve for each item's [V[j], b_j].
# Each sub-problem is solved EXACTLY in closed form, so the regularised
# loss L can never go up from one half-step to the next — L is monotone
# non-increasing by construction. The plain training RMSE is NOT
# guaranteed to fall every iteration (the penalty term can trade a little
# fit for smaller weights), so we track L for the convergence check.
#
# Why regularise hard? Each user has only ~25 training ratings. Without a
# strong lambda, k factors per user can memorise those ratings (tiny train
# RMSE) and generalise terribly (large holdout RMSE) — overfitting.
#
# Connection to SVD: when R is fully observed and lambda = 0, ALS
# converges to the truncated SVD. Sparse + regularised ALS is a GENERALISED
# SVD that handles missing values gracefully.



## TASK 2 — BUILD the ALS update step


Returns a dict with U, V, b_user, b_item, mu, predictions,
    loss_history (regularised objective) and rmse_history (train RMSE).
    Items with no training ratings get NaN predictions: ALS has nothing to
    place them with.


In [ ]:
def als_matrix_factorisation(
    R: np.ndarray,
    obs_mask: np.ndarray,
    k: int,
    lam: float,
    n_iter: int,
    rng: np.random.Generator,
) -> dict:
    n_users, n_items = R.shape
    mu = float(R[obs_mask].mean())
    U = rng.normal(0, 0.1, size=(n_users, k))
    V = rng.normal(0, 0.1, size=(n_items, k))
    b_user = np.zeros(n_users)
    b_item = np.zeros(n_items)
    R_safe = np.nan_to_num(R, nan=0.0)
    penalty = lam * np.eye(k + 1)
    loss_history: list[float] = []
    rmse_history: list[float] = []

    for iteration in range(n_iter):
        # Fix the item side, solve a ridge regression for each user.
        # Design matrix = [V_j, 1] so the solve returns [U[u], b_u].
        for u in range(n_users):
            rated = np.where(obs_mask[u])[0]
            if len(rated) == 0:
                continue
            X_u = np.hstack([V[rated], np.ones((len(rated), 1))])
            y_u = R_safe[u, rated] - mu - b_item[rated]
            A = X_u.T @ X_u + penalty
            b = X_u.T @ y_u
            solution = np.linalg.solve(A, b)
            U[u], b_user[u] = solution[:k], solution[k]

        # Fix the user side, solve a ridge regression for each item.
        for j in range(n_items):
            raters = np.where(obs_mask[:, j])[0]
            if len(raters) == 0:
                continue
            X_j = np.hstack([U[raters], np.ones((len(raters), 1))])
            y_j = R_safe[raters, j] - mu - b_user[raters]
            A = X_j.T @ X_j + penalty
            b = X_j.T @ y_j
            solution = np.linalg.solve(A, b)
            V[j], b_item[j] = solution[:k], solution[k]

        R_hat = mu + b_user[:, None] + b_item[None, :] + U @ V.T
        residuals = (R_safe - R_hat)[obs_mask]
        loss = float(
            (residuals**2).sum()
            + lam * ((U**2).sum() + (V**2).sum() + (b_user**2).sum() + (b_item**2).sum())
        )
        rmse = float(np.sqrt(np.mean(residuals**2)))
        loss_history.append(loss)
        rmse_history.append(rmse)
        if iteration % 5 == 0 or iteration == n_iter - 1:
            print(f"  iter {iteration:3d}: loss = {loss:10.2f}  train RMSE = {rmse:.4f}")

    predictions = np.clip(mu + b_user[:, None] + b_item[None, :] + U @ V.T, 1.0, 5.0)
    predictions[:, obs_mask.sum(axis=0) == 0] = np.nan  # unseen items
    return {
        "U": U,
        "V": V,
        "b_user": b_user,
        "b_item": b_item,
        "mu": mu,
        "predictions": predictions,
        "loss_history": loss_history,
        "rmse_history": rmse_history,
    }



## TASK 3 — TRAIN via ALS


In [ ]:
print("\n" + "=" * 70)
print("  ALS Matrix Factorisation on SG E-commerce Ratings")
print("=" * 70)
print(f"k={K_LATENT}, lambda={LAMBDA_REG}, iterations={N_ITERATIONS}")

data = build_rating_dataset()
R_train = data["R_train"]
R_observed = data["R_observed"]
train_mask = data["train_mask"]
holdout_mask = data["holdout_mask"]
U_true = data["U_true"]
V_true = data["V_true"]

als = als_matrix_factorisation(
    R_train,
    train_mask,
    k=K_LATENT,
    lam=LAMBDA_REG,
    n_iter=N_ITERATIONS,
    rng=data["rng"],
)
U_learned, V_learned = als["U"], als["V"]
loss_history, rmse_history = als["loss_history"], als["rmse_history"]
als_predictions = als["predictions"]
als_rmse, als_cov = holdout_rmse(als_predictions, R_observed, holdout_mask)



### Checkpoint


In [ ]:
assert U_learned.shape == (N_USERS, K_LATENT), "U must be (N_USERS, K_LATENT)"
assert V_learned.shape == (N_ITEMS, K_LATENT), "V must be (N_ITEMS, K_LATENT)"
for i in range(1, len(loss_history)):
    assert loss_history[i] <= loss_history[i - 1] * (1 + 1e-9), (
        f"Regularised ALS loss must be non-increasing (violated at step {i})"
    )
print(
    f"\n[ok] Checkpoint passed — ALS loss non-increasing: "
    f"{loss_history[0]:.1f} -> {loss_history[-1]:.1f}; train RMSE "
    f"{rmse_history[-1]:.4f}, holdout RMSE={als_rmse:.4f}, coverage={als_cov:.0%}\n"
)



## TASK 4 — VISUALISE embeddings + convergence


1.0 = identical subspaces. Uses ALL columns of both matrices, so the
    arbitrary rotation/order of learned factors does not matter.


In [ ]:
# The 5-dimensional learned factors can be projected to 2D via PCA. The
# result is a scatter plot where geometric proximity = similar taste.

viz = ModelVisualizer()

pca_users = PCA(n_components=2, random_state=42)
U_2d = pca_users.fit_transform(U_learned)
user_df = pl.DataFrame(
    {
        "user_id": data["user_ids"],
        "pc1": U_2d[:, 0].tolist(),
        "pc2": U_2d[:, 1].tolist(),
        "learned_user_bias": als["b_user"].tolist(),
    }
)
fig_u = viz.scatter(user_df, x="pc1", y="pc2", color="learned_user_bias")
fig_u.update_layout(title="ALS User Embeddings (2D PCA projection)")
save_html(fig_u, "04_als_user_embeddings.html")

pca_items = PCA(n_components=2, random_state=42)
V_2d = pca_items.fit_transform(V_learned)
item_df = pl.DataFrame(
    {
        "item_id": data["item_ids"],
        "pc1": V_2d[:, 0].tolist(),
        "pc2": V_2d[:, 1].tolist(),
        "learned_item_bias": als["b_item"].tolist(),
    }
)
fig_i = viz.scatter(item_df, x="pc1", y="pc2", color="learned_item_bias")
fig_i.update_layout(title="ALS Item Embeddings (2D PCA projection)")
save_html(fig_i, "04_als_item_embeddings.html")

fig_conv = viz.training_history(
    {"regularised_loss": loss_history}, x_label="ALS iteration"
)
fig_conv.update_layout(title="ALS Convergence (regularised objective)")
save_html(fig_conv, "04_als_convergence.html")


# Subspace alignment — did ALS recover the true latent structure?
def subspace_similarity(A: np.ndarray, B: np.ndarray) -> float:
    Qa, _ = np.linalg.qr(A)
    Qb, _ = np.linalg.qr(B)
    sigmas = np.linalg.svd(Qa.T @ Qb, compute_uv=False)
    return float(np.mean(np.minimum(sigmas, 1.0)))


warm_items = ~data["cold_items"]  # cold SKUs were never trained
user_align = subspace_similarity(U_learned, U_true)
item_align = subspace_similarity(V_learned[warm_items], V_true[warm_items])
chance_rng = np.random.default_rng(0)
user_chance = subspace_similarity(chance_rng.normal(size=U_true.shape), U_true)
item_chance = subspace_similarity(
    chance_rng.normal(size=V_true[warm_items].shape), V_true[warm_items]
)
print(
    f"\nSubspace recovery (1.0 = perfect): user={user_align:.3f} "
    f"(random-subspace baseline {user_chance:.3f}), item={item_align:.3f} "
    f"(baseline {item_chance:.3f})"
)
recovered = user_align > user_chance + 0.3 and item_align > item_chance + 0.3
if recovered:
    print(
        "  -> ALS found the hidden taste space far better than chance, "
        "without ever being told what the factors mean."
    )
else:
    print(
        "  -> Recovery is close to the random baseline on this run — the "
        "factors fit the ratings but do NOT match the true taste space."
    )

print_baselines(R_train, train_mask, R_observed, holdout_mask)
print_method_scores("ALS MF", als_predictions, R_observed, holdout_mask)
print_warm_comparison(
    "ALS MF", als_predictions, R_train, train_mask, R_observed, holdout_mask,
    data["cold_items"],
)
overfit_gap = als_rmse - rmse_history[-1]
print(
    f"  Train RMSE {rmse_history[-1]:.4f} vs holdout RMSE {als_rmse:.4f} "
    f"(gap {overfit_gap:+.4f}). A large gap would mean lambda is too small."
)



## SVD++ — conceptual extension (no code, just the intuition)


In [ ]:
# Plain MF: r_hat(u, i) = U[u] · V[i]
#
# SVD++ adds implicit feedback (clicks, views, not just ratings):
#
#   r_hat(u, i) = mu + b_u + b_i
#               + V[i] · (U[u] + |N(u)|^(-1/2) * sum_{j in N(u)} y[j])
#
# where:
#   mu, b_u, b_i  = global/user/item biases
#   N(u)          = items user u interacted with (implicit set)
#   y[j]          = implicit feedback vectors
#
# The term after U[u] is "if you clicked these items, your taste drifts
# in this direction." It captures information that pure rating-based MF
# misses. SVD++ (Koren, KDD 2008) was one of the core models in the
# BellKor team's Netflix Prize blends.



## TASK 5 — APPLY: Music-Streaming Recommendation at Scale


In [ ]:
# SCENARIO: A Southeast Asian music streaming service has 600M "plays"
# per day across 80M tracks and 12M users.
# Explicit ratings (thumbs up/down) are rare — <3% of plays. Implicit
# feedback (skipped after 5s, played to completion, added to playlist) is
# the only usable signal at volume.
#
# Why matrix factorisation is the right tool:
#   - Implicit feedback is ~97% of the signal — user-CF and item-CF
#     struggle with binary play/skip data; MF with a confidence weighting
#     (Hu et al. 2008) handles it natively
#   - Embeddings compress 12M x 80M = 960T pairs down to (12M + 80M) x
#     128 float32 numbers ~ 47GB — big, but it fits in the RAM of one
#     large server, unlike the 960-trillion-cell matrix
#   - Once trained, scoring a (user, track) pair is a single 128-number
#     dot product — cheap enough to score thousands of candidates per
#     request
#   - The same embeddings can power "similar tracks", a personalised daily
#     mix and a weekly discovery playlist — train once, serve several
#     products
#
# BUSINESS IMPACT (illustrative assumptions, not measured figures): on a
# market with S$180M annual streaming revenue, if better discovery
# recommendations cut churn enough to retain 2% of revenue, that is
# ~S$3.6M/year. The size of the real effect has to be measured with an
# online A/B test; the offline holdout metrics above only tell you whether
# ALS ranks held-out items better than the baselines.
#
# LIMITATIONS:
#   - Cold-start tracks: a brand-new upload has no listens = no factors
#     (solution: hybrid with content-based audio embeddings)
#   - Popularity bias: top-streamed tracks dominate every recommendation
#   - Filter bubble: a low-dimensional taste space collapses the long tail
#
# THE PIVOT: optimisation drives feature discovery.
#
#   We never told ALS what the 5 latent dimensions mean. It learned them
#   by minimising reconstruction error. Compare:
#       MF:              min ||R - U V^T||^2
#       Neural net:      min ||y - f(W x + b)||^2
#   SAME PRINCIPLE. Neural nets just add non-linearities.
#
# The next technique (05_hybrid_evaluation.py) combines all four
# recommenders. ALS cannot score the brand-new SKUs (no ratings, no
# factors) — a hybrid can route those to content-based filtering.



## REFLECTION


[x] Implemented biased ALS from scratch — ridge regression per user + per item
  [x] Verified the regularised loss is non-increasing ({loss_history[0]:.1f} -> {loss_history[-1]:.1f})
  [x] Measured subspace recovery against a random baseline
      (user {user_align:.3f} vs {user_chance:.3f}, item {item_align:.3f} vs {item_chance:.3f})
  [x] Visualised 5-dim embeddings in 2D via PCA
  [x] Understood SVD++ as the implicit-feedback extension of MF
  [x] Sized an (illustrative) SEA streaming scenario for MF

  THE PIVOT: Matrix factorisation DISCOVERS latent factors by optimising
  a loss. This is the same principle as neural network training — the
  hidden layer is just an embedding, learned by backpropagation.

  Next: 05_hybrid_evaluation.py — combine all four recommenders and see
  why Netflix Prize teams won by blending many models.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

